# Full-run test for SMOTE-NC

This notebook provides runnable cells to perform full training runs for SMOTE-NC using the project TrainTestSplitPipeline.

SMOTE-NC handles mixed categorical and numerical features.

In [1]:
pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports and helpers
import os
import importlib
import traceback
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess

def ensure(path):
    os.makedirs(path, exist_ok=True)

# Convenience wrapper to create a pipeline that optionally disables evaluations
def make_pipeline(model_callable, skip_evaluations=True, evaluations=None):
    if skip_evaluations:
        return TrainTestSplitPipeline(model=model_callable, evaluations=[], override_evaluations=True)
    elif evaluations is not None:
        return TrainTestSplitPipeline(model=model_callable, evaluations=evaluations, override_evaluations=True)
    else:
        return TrainTestSplitPipeline(model=model_callable)

In [3]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [4]:
# User configuration: choose dataset(s) and run options
DATASETS = ['adult', 'car', 'magic', 'nursery', 'shuttle']
SKIP_EVALUATIONS = True

# Top-level dirs
ensure('discretized_data')
ensure('sample_data')
ensure('synthetic')
ensure('Results')

MODEL_MAP = {
    'smotenc': ('katabatic.models.smotenc.models', 'SMOTENCModel'),
}

SMOTENC_CONFIG = {
    'k_neighbors': 5,
    'sampling_strategy': 'auto',
    'random_state': 42,
    'categorical_features': None,  # Auto-detect, or specify list like [0, 1, 3]
}

## Preprocess datasets (run once)
Run this cell to discretize the raw CSVs into `discretized_data/{dataset}.csv`. 

In [5]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'Preprocessing {dataset}...')
    try:
        discretize_preprocess(file_path=f'raw_data/{dataset}.csv', output_path=f'discretized_data/{dataset}.csv', bins=10, strategy='uniform')
        print(f'Discretized -> discretized_data/{dataset}.csv')
    except Exception as e:
        print(f'Failed to preprocess {dataset}: {e}')
        import traceback; traceback.print_exc()


Preprocessing adult...
Preprocessing: raw_data/adult.csv
Saved preprocessed discrete dataset to: discretized_data/adult.csv
Discretized -> discretized_data/adult.csv

Preprocessing car...
Preprocessing: raw_data/car.csv
Saved preprocessed discrete dataset to: discretized_data/car.csv
Discretized -> discretized_data/car.csv

Preprocessing magic...
Preprocessing: raw_data/magic.csv
Saved preprocessed discrete dataset to: discretized_data/magic.csv
Discretized -> discretized_data/magic.csv

Preprocessing nursery...
Preprocessing: raw_data/nursery.csv
Saved preprocessed discrete dataset to: discretized_data/nursery.csv
Discretized -> discretized_data/nursery.csv

Preprocessing shuttle...
Preprocessing: raw_data/shuttle.csv
Saved preprocessed discrete dataset to: discretized_data/shuttle.csv
Discretized -> discretized_data/shuttle.csv


## Run full SMOTE-NC
This cell runs CTGAN for each selected dataset using `CTGAN_CONFIG` above. Be patient — full training can take time depending on `epochs` and dataset size.

In [ ]:
for dataset in DATASETS:
    print('\n' + '='*60)
    print(f'SMOTE-NC -> {dataset}')
    synth_dir = os.path.join('synthetic', dataset, 'smotenc')
    ensure(synth_dir)
    try:
        mod_path, cls_name = MODEL_MAP['smotenc']
        module = importlib.import_module(mod_path)
        ModelClass = getattr(module, cls_name)
        model_factory = lambda: ModelClass(**SMOTENC_CONFIG)
        pipeline = make_pipeline(model_factory, skip_evaluations=SKIP_EVALUATIONS)
        result = pipeline.run(
            input_csv=f'discretized_data/{dataset}.csv',
            output_dir=f'sample_data/{dataset}',
            synthetic_dir=synth_dir,
            real_test_dir=f'sample_data/{dataset}'
        )
        print('SMOTE-NC finished for', dataset)
    except Exception as e:
        print('SMOTE-NC failed for', dataset, e)
        import traceback; traceback.print_exc()


SMOTE-NC -> adult
Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[SMOTE-NC] Auto-detected 11 categorical features
[SMOTE-NC] Categorical feature indices: [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12]
[SMOTE-NC] Initializing with k_neighbors=5...
[SMOTE-NC] Ready to generate samples from 26048 training samples...
[SMOTE-NC] Generated samples in 6.81 seconds.
[SMOTE-NC] Generated 13502 new synthetic samples...
[SMOTE-NC] Returning 26048 total samples (original size with balanced classes)...
[SMOTE-NC] Synthetic data saved:
  X -> synthetic\adult\smotenc\x_synth.csv
  y -> synthetic\adult\smotenc\y_synth.csv
